In [ ]:
# Set up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
sys.path.append(parent_dir)
print('Parent directory set to:', parent_dir)

In [ ]:
# Import general packages
import torch
import numpy as np
from itertools import product

# Import CRN packages
from RL4CRN.iocrns.iocrn import IOCRN
from RL4CRN.utils.ic import IC

In [ ]:
# Flag to resimulate the IOCRN dynamics
resimulate_flag = False  

In [ ]:
# Define path for halls of fame directory
base_path_hof = os.path.abspath(os.path.join(os.getcwd(), '..', 'hof'))
print(f"Current working directory: {os.getcwd()}")
print(f"Base path for halls of fame: {base_path_hof}")

In [ ]:
# Get list of hall of fame files
hof_files = [f for f in os.listdir(base_path_hof) if f.startswith('hall_of_fame_') and f.endswith('.pth')]
print(f"Found {len(hof_files)} hall of fame files:")
for file in hof_files:
    print(f"- {file}")

In [ ]:
# Load the list of IOCRNs from each hall of fame file
hof_data = {}
for file in hof_files:
    file_path = os.path.join(base_path_hof, file)
    hof_data[file] = torch.load(file_path, weights_only=False)
    print(f"Loaded {file} with {len(hof_data[file])} entries.")

# Collect all halls of fame into a single list
hof_iocrns = []
for file, iocrn_list in hof_data.items():
    hof_iocrns.extend(iocrn_list)
print("-" * 40)
print(f"Total IOCRNs collected from all halls of fame: {len(hof_iocrns)}")

In [ ]:
# Construct the IOCRN inputs
nums = [0.5, 1.0, 1.5]
u_list = [np.array(u) for u in product(nums, repeat=hof_iocrns[0].num_inputs)] # list of input combinations, each input is a numpy array of shape (p,)

# Construct the IOCRN initial conditions
ic = IC(names=hof_iocrns[0].species_labels, values=[[0.0, 0.0, 0.0]])

In [ ]:
print(hof_iocrns[0].last_task_info['reward'])
hof_iocrns[0].plot_transient_response()

In [ ]:
# Reset all IOCRNs
for iocrn in hof_iocrns:
    iocrn.reset()
    
# Set simulation parameters
t_f = 200                                                   # Final time for the simulation
N_t = 1000                                                  # Number of time steps
time_horizon = np.linspace(0, t_f, N_t, dtype=np.float32)

# Simulate each IOCRN
x0_list = ic.get_ic(hof_iocrns[0])
t, x, y, task_info = hof_iocrns[0].transient_response(u_list, x0_list, time_horizon, LARGE_NUMBER=1e4)
hof_iocrns[0].plot_transient_response()